# 01. Exploratory Data Analysis (EDA)

Анализ данных датасета CEAS_08 для обнаружения фишинговых писем.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

## Загрузка данных

In [ ]:
df = pd.read_csv('data/raw/CEAS_08.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## Информация о данных

In [ ]:
df.info()

## Распределение классов

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='label', data=df, palette='viridis')
plt.title('Распределение классов (0 - легитимное, 1 - фишинг)')
plt.xlabel('Класс')
plt.ylabel('Количество')
plt.xticks([0, 1], ['Легитимное', 'Фишинг'])
plt.show()

print(f"\nClass distribution:\n{df['label'].value_counts()}")
print(f"\nClass balance: {df['label'].value_counts(normalize=True).round(3).to_dict()}")

## Анализ текста писем

In [ ]:
df['text_length'] = df['body'].fillna('').apply(len)
df['word_count'] = df['body'].fillna('').apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x='text_length', hue='label', bins=50, ax=axes[0])
axes[0].set_title('Распределение длины текста')
axes[0].set_xlabel('Длина текста')

sns.histplot(data=df, x='word_count', hue='label', bins=50, ax=axes[1])
axes[1].set_title('Распределение количества слов')
axes[1].set_xlabel('Количество слов')

plt.tight_layout()
plt.show()

## Анализ URL в письмах

In [ ]:
import re

def count_urls(text):
    urls = re.findall(r'http\S+|www\.\S+', str(text))
    return len(urls)

df['url_count'] = df['body'].apply(count_urls)

plt.figure(figsize=(8, 5))
sns.boxplot(x='label', y='url_count', data=df)
plt.title('Количество URL по классам')
plt.xlabel('Класс')
plt.ylabel('Количество URL')
plt.xticks([0, 1], ['Легитимное', 'Фишинг'])
plt.show()

## WordCloud для фишинговых писем

In [ ]:
phishing_text = ' '.join(df[df['label'] == 1]['body'].dropna().astype(str))

wordcloud = WordCloud(width=800, height=400, background_color='white').generate(phishing_text)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('WordCloud для фишинговых писем')
plt.show()

## Домены отправителей

In [ ]:
def extract_domain(sender):
    if pd.isna(sender):
        return ''
    match = re.search(r'@([\w.-]+)', sender)
    return match.group(1) if match else ''

df['sender_domain'] = df['sender'].apply(extract_domain)

top_domains = df[df['label'] == 1]['sender_domain'].value_counts().head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_domains.values, y=top_domains.index)
plt.title('Топ-15 доменов отправителей фишинговых писем')
plt.xlabel('Количество')
plt.ylabel('Домен')
plt.show()